In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
import sys

sys.path.append("../..")

import json
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns
import xarray as xr
import dask
from dask.diagnostics import ProgressBar

import pvlib
from src.conversion_energy.pv import (
    DEFAULT_PV_CONFIGURATION,
    NORMALISED_ROOFTOP_PV_CONFIGURATION,
    ROOFTOP_PV_CONFIGURATION,
    compute_pv_for_grid
)
from src.data_processing.load_data import ClimateModelData, ClimateReanalysisData
from src.utils.utils_xarray import (
    get_time_resolution,
    match_instaneous_to_mean_values,
    match_time_steps,
    match_time_steps_interpolate,
    sel_extent,
)

from src.utils.constants import BC_SUBDAILY_CHUNKS
sns.set_style("whitegrid", {"grid.color": ".93"})

In [7]:
dask.config.set(num_workers=15, scheduler="processes")

dask.config.set({
    "distributed.worker.memory.target": 0.7,  # Spill to disk at 70%
    "distributed.worker.memory.spill": 0.8,   # Start spilling at 80%
    "distributed.worker.memory.pause": 0.9,   # Pause worker at 90%
    "array.chunk-size": "512MiB",
    "logging.distributed": "error",           # Less verbose logs
})

In [8]:
paths = json.load(open("../../paths_so4.json"))

path_data_analysis = Path(paths["path_data_analysis"])
path_data_processed = Path(paths["path_data_processed"])
path_data_cordex = Path(paths["path_data_cordex"])
path_data_adjusted_cordex = Path(paths["path_data_adjusted_cordex"])
path_data_cerra = Path(paths["path_data_cerra"])
path_data_cerra_land = Path(paths["path_data_cerra_land"])
path_data_personal = Path(paths["path_data_personal"])
path_data_personal_cordex = path_data_personal / "cordex"

path_energy_ch = path_data_processed / "energy" / "ch"

path_figs = Path(paths["path_figs"])

In [9]:
climate_model_chain = "CNRM-ALADIN63_CNRM-CERFACS-CNRM-CM5_r1i1p1"
climate_scenario = "rcp85"

In [10]:
extent = {"lat_min": 30, "lat_max": 75, "lon_min": -12, "lon_max": 30}
extent_ch = {"lat_min": 45.8, "lat_max": 47.9, "lon_min": 5.9, "lon_max": 10.5}

# Load data

In [ ]:
ds_rsds_bc = ClimateModelData(
    path_data_processed
    / "cordex_processed"
    / f"{climate_model_chain}_{climate_scenario}"
    / "bias_corrected"
    / "3hr"
    / "rsds.zarr",
    extent=extent_ch,
    chunks=BC_SUBDAILY_CHUNKS,
)

ds_tas_bc = ClimateModelData(
    path_data_processed
    / "cordex_processed"
    / f"{climate_model_chain}_{climate_scenario}"
    / "bias_corrected"
    / "3hr"
    / "tas.zarr",
    extent=extent_ch,
    chunks=BC_SUBDAILY_CHUNKS
)

ds_sfcWind_bc = ClimateModelData(
    path_data_processed
    / "cordex_processed"
    / f"{climate_model_chain}_{climate_scenario}"
    / "bias_corrected"
    / "3hr"
    / "sfcWind.zarr",
    extent=extent_ch,
    chunks=BC_SUBDAILY_CHUNKS
)

ds_orog = sel_extent(
    xr.open_dataset(
        sorted(
            (
                path_data_processed
                / "cordex_processed"
                / f"{climate_model_chain}_{climate_scenario}"
                / "regridded"
                / "fx"
            ).glob("*.nc")
        )[0]
    ),
    extent_ch,
)

In [12]:
time_slice = slice("1991", "1993")
ds_tas_bc.ds_data = ds_tas_bc.ds_data.sel(time=time_slice)
ds_sfcWind_bc.ds_data = ds_sfcWind_bc.ds_data.sel(time=time_slice)
ds_rsds_bc.ds_data = ds_rsds_bc.ds_data.sel(time=time_slice)

print("Matching instantaneous to mean values")

with ProgressBar(dt=1):
    ds_tas_bc.ds_data = match_instaneous_to_mean_values(
        ds_tas_bc.ds_data - 273.15, skip_leap_days=True, time_res="3h"
    ).load()
    ds_tas_bc.ds_data["time"] = ds_tas_bc.ds_data["time"] + pd.Timedelta(
        hours=1, minutes=30
    )

    ds_sfcWind_bc.ds_data = match_instaneous_to_mean_values(
        ds_sfcWind_bc.ds_data, skip_leap_days=True, time_res="3h"
    ).load()
    ds_sfcWind_bc.ds_data["time"] = ds_sfcWind_bc.ds_data["time"] + pd.Timedelta(
        hours=1, minutes=30
    )


Matching instantaneous to mean values
[########################################] | 100% Completed | 60.23 s
[########################################] | 100% Completed | 51.06 s


In [15]:
tilt = 35
azimuth = 180
direction = "S"

ds_pv = (
    compute_pv_for_grid(
        ds_rsds_bc.ds_data.rsds,
        ds_tas_bc.ds_data.tas,
        ds_sfcWind_bc.ds_data.sfcWind,
        tilt=tilt,
        azimuth=azimuth,
        pv_configuration=NORMALISED_ROOFTOP_PV_CONFIGURATION,
        da_altitude=ds_orog.orog,
        num_workers=15,
    ).rename({"ac_power": f"tilt_{tilt}_azimuth_{direction}"})
    / NORMALISED_ROOFTOP_PV_CONFIGURATION["module_parameters"]["pdc0"]
)

KeyboardInterrupt: 